# Colab bootstrap

Verifies the Colab environment: repository checkout, ngspice installation, package import, and agreement of the Colab ngspice result for the nominal design with the stored reference (`results/reference_nominal.json`).

Requirements: Colab runtime with Drive mounted; code is pulled from this repository.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, subprocess
REPO = "/content/spice-verified-analog-finetuning"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Keerthanareddy17/spice-verified-analog-finetuning.git", REPO], check=True)
os.chdir(REPO)

In [3]:
!apt-get -qq install -y ngspice > /dev/null
!ngspice --version | head -3
!pip -q install -e .

******
** ngspice-42 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for analogft (pyproject.toml) ... done


In [8]:
import sys, importlib, os
REPO = "/content/spice-verified-analog-finetuning"
print(os.listdir(REPO))                      # expect 'src'
print(os.listdir(f"{REPO}/src"))             # expect 'analogft'
sys.path.insert(0, f"{REPO}/src")
importlib.invalidate_caches()
import analogft
print("analogft imported from:", analogft.__file__)

['src', 'LICENSE', 'results', 'data', 'docs', 'models', '.env.example', 'notebooks', 'runs', 'scripts', 'tests', 'pyproject.toml', '.gitignore', '.git', 'README.md']
['analogft', 'analogft.egg-info']
analogft imported from: /content/spice-verified-analog-finetuning/src/analogft/__init__.py


In [9]:
import json
from analogft import paths
from analogft.circuit import nominal_design, simulate
print("IS_COLAB:", paths.IS_COLAB, "| STORAGE_ROOT:", paths.STORAGE_ROOT)

ref = json.load(open(paths.REPO_ROOT / "results" / "reference_nominal.json"))
r = simulate(nominal_design(), use_cache=False)
print("Reference ngspice:", ref["ngspice_version"], "| Colab ngspice:", r.ngspice_version)
for k, v in ref["metrics"].items():
    c = r.metrics[k]
    rel = abs(c - v) / max(abs(v), 1e-30)
    print(f"{k:<11} reference={v:<14.6g} colab={c:<14.6g} rel.diff={rel:.2e}", "OK" if rel < 1e-2 else "MISMATCH")
assert r.ok and all(abs(r.metrics[k]-v)/max(abs(v),1e-30) < 1e-2 for k, v in ref["metrics"].items()), "Colab ngspice disagrees with the stored reference"
print("Colab ngspice agrees with the stored reference (within 1%).")

IS_COLAB: True | STORAGE_ROOT: /content/drive/MyDrive/spice-verified-analog-finetuning
Reference ngspice: 47 | Colab ngspice: 42
power_w     reference=0.000436248    colab=0.000436248    rel.diff=0.00e+00 OK
vout_dc     reference=0.899975       colab=0.899975       rel.diff=0.00e+00 OK
dc_gain_db  reference=74.205         colab=74.205         rel.diff=0.00e+00 OK
ugf_hz      reference=2.787e+07      colab=2.787e+07      rel.diff=0.00e+00 OK
pm_deg      reference=67.7666        colab=67.7666        rel.diff=0.00e+00 OK
Colab ngspice agrees with the stored reference (within 1%).


In [10]:
# Manifest check: data/ comes from the repository checkout, runs/ and results/ from Drive.
from analogft.drive_check import verify_manifest
for name, root in (("data", paths.DATA_DIR), ("runs", paths.RUNS_DIR), ("results", paths.RESULTS_DIR)):
    if not (root / "manifest.json").exists():
        print(name, ": no manifest"); continue
    verify_manifest(root, raise_on_error=True)
    print(name, ": complete")

data : no manifest yet
runs : no manifest yet
results : complete
